# K-Means — finding groups nobody labelled

## A different kind of problem

Every model in the Supervised ML module learned from **labelled** examples: here are the
penguins, here are their species, learn the mapping.

Now the labels are gone. You have measurements and nothing else, and the question is whether
the data falls into natural groups at all.

That is **unsupervised learning**, and it changes something fundamental: there is no correct
answer to check against. You cannot compute accuracy, because there is nothing to be accurate
about. Most of this lesson is about what to do instead.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RSEED = 42

## The algorithm

K-means is four steps, repeated until nothing moves:

1. Drop `k` points at random — the **centroids**.
2. Assign every observation to its nearest centroid.
3. Move each centroid to the mean of the points assigned to it.
4. Go to 2.

It is minimising one quantity: the total squared distance from each point to its own
centroid. scikit-learn calls this **inertia**.

$$\text{inertia} = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2$$

Watch it converge on some obviously-grouped synthetic data.

In [ ]:
X_blobs, true_labels = make_blobs(
    n_samples=400, centers=4, cluster_std=1.1, random_state=RSEED
)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, n_iter in zip(axes, [1, 2, 3, 20]):
    km = KMeans(n_clusters=4, max_iter=n_iter, n_init=1, init="random", random_state=1).fit(X_blobs)
    ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=km.labels_, cmap="viridis", s=12, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c="#F5821F", s=180, marker="X", edgecolor="black", linewidth=1.2)
    ax.set_title(f"after {n_iter} iteration{'s' if n_iter > 1 else ''}\ninertia {km.inertia_:.0f}", fontsize=10)
plt.tight_layout()
plt.show()

The centroids walk into position and the assignment stops changing.

## `n_init` — because the starting points matter

K-means finds a *local* minimum. Start it somewhere unlucky and it converges to a bad answer
and stays there.

scikit-learn's defence is to run the whole thing several times from different starts and keep
the best, controlled by `n_init`. The default is 10 — leave it alone unless you have a
reason.

In [ ]:
for n_init in [1, 10]:
    inertias = [
        KMeans(n_clusters=4, n_init=n_init, init="random", random_state=s).fit(X_blobs).inertia_
        for s in range(20)
    ]
    print(f"n_init={n_init:<3} inertia over 20 seeds: "
          f"best {min(inertias):.0f}, worst {max(inertias):.0f}, spread {max(inertias) - min(inertias):.0f}")

With a single start the answer you get depends on luck. With ten it is stable.

## The assumption that will bite you

K-means draws **spherical** clusters of **roughly equal size**, because "nearest centroid"
carves space into straight-edged regions around each centre.

When the real groups are not shaped like that, it fails — and it fails silently, returning
confident labels that are simply wrong.

In [ ]:
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=400, noise=0.06, random_state=RSEED)
X_uneven, _ = make_blobs(n_samples=[400, 60, 60], centers=[[0, 0], [5, 5], [-5, 5]],
                         cluster_std=[2.2, 0.4, 0.4], random_state=RSEED)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (data, title) in zip(axes, [(X_moons, "two crescents"), (X_uneven, "unequal sizes and spreads")]):
    labels = KMeans(n_clusters=2 if data is X_moons else 3, n_init=10, random_state=RSEED).fit_predict(data)
    ax.scatter(data[:, 0], data[:, 1], c=labels, cmap="viridis", s=14, alpha=0.75)
    ax.set_title(f"k-means on {title}")
plt.tight_layout()
plt.show()

Neither is what a person would draw. K-means cut the crescents straight down the middle,
and split the one big diffuse blob rather than keeping it whole.

**These are not bugs.** They are the algorithm doing exactly what it optimises for. When your
groups are elongated, nested or wildly different in density, you need a different algorithm —
which is what the HDBSCAN notebook is for.

## Scaling is not optional

K-means measures distance, and distance is meaningless across features on different scales —
the same argument as for KNN in the Supervised ML module, and with the same fix.

On the penguins, body mass is measured in thousands of grams and bill depth in tens of
millimetres. Cluster without scaling and body mass decides everything.

In [ ]:
penguins = pd.read_csv("data/penguins.csv").dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
)
measurements = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

print(measurements.std().round(1).to_dict())

In [ ]:
unscaled = KMeans(n_clusters=3, n_init=10, random_state=RSEED).fit(measurements)
scaled = make_pipeline(
    StandardScaler(), KMeans(n_clusters=3, n_init=10, random_state=RSEED)
).fit(measurements)

comparison = pd.DataFrame({
    "unscaled": pd.Series(unscaled.labels_).value_counts().sort_index(),
    "scaled": pd.Series(scaled[-1].labels_).value_counts().sort_index(),
})
print(comparison, "\n")

# how much does each cluster differ in body mass alone?
for name, labels in [("unscaled", unscaled.labels_), ("scaled", scaled[-1].labels_)]:
    spread = penguins.groupby(labels)["body_mass_g"].mean()
    print(f"{name:9s} cluster mean body mass: {spread.round(0).to_dict()}")

Without scaling the clusters are essentially a three-way split of body mass — the other
three measurements contributed almost nothing, because their numbers are small. With scaling
every measurement gets a say.

**Put `StandardScaler` in a pipeline in front of every distance-based algorithm.** Same rule
as KNN.

## Summary

- K-means alternates *assign to nearest centroid* and *move centroid to the mean*, minimising
  **inertia**.
- It finds a local minimum, so `n_init` restarts matter — the default of 10 is fine.
- It assumes **spherical, similarly sized** clusters, and fails silently when that is wrong.
- **Always scale first.**

> **Next:** we picked `k = 3` because we happen to know there are three penguin species. In
> a real clustering problem nobody tells you `k`.